In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# PATHS
# =========================
BASE_DIR = Path.cwd()
if BASE_DIR.name == "scripts":
    BASE_DIR = BASE_DIR.parent

input_path = BASE_DIR / "data/factsheet/NFHS_5_India_Districts_Factsheet_Data.xls"
output_path = BASE_DIR / "data/variables/nfhs_ncd_pct.csv"

print("Input:", input_path)
print("Exists:", input_path.exists())

# =========================
# LOAD DATA
# =========================
df = pd.read_excel(input_path)

# =========================
# AUTO-DETECT DISTRICT COLUMN
# =========================
district_candidates = [c for c in df.columns if "district" in str(c).lower()]
if not district_candidates:
    raise ValueError("No district column found")

district_col = district_candidates[0]
print("Using district column:", district_col)

# =========================
# CLEAN DISTRICT VALUES
# =========================
df[district_col] = (
    df[district_col]
    .astype(str)
    .str.strip()
    .str.replace("\n", "", regex=True)
    .str.replace("\t", "", regex=True)
    .str.replace("\xa0", "", regex=True)
)

# =========================
# NFHS → STANDARD DISTRICT MAP
# =========================
district_map = {

    # Direct matches
    "Kokrajhar": "Kokrajhar",
    "Goalpara": "Goalpara",
    "Barpeta": "Barpeta",
    "Morigaon": "Morigaon",
    "Lakhimpur": "Lakhimpur",
    "Dhemaji": "Dhemaji",
    "Tinsukia": "Tinsukia",
    "Dibrugarh": "Dibrugarh",
    "Golaghat": "Golaghat",
    "Dima Hasao": "Dima Hasao",
    "Cachar": "Cachar",
    "Karimganj": "Karimganj",
    "Hailakandi": "Hailakandi",
    "Bongaigaon": "Bongaigaon",
    "Chirang": "Chirang",
    "Kamrup": "Kamrup",
    "Kamrup Metropolitan": "Kamrup Metropolitan",
    "Kamrup Metro": "Kamrup Metropolitan",
    "Nalbari": "Nalbari",
    "Baksa": "Baksa",
    "Darrang": "Darrang",
    "Udalguri": "Udalguri",
    "Biswanath": "Biswanath",
    "Biswanath Chariali": "Biswanath",
    "Charaideo": "Charaideo",
    "Dhubri": "Dhubri",
    "Hojai": "Hojai",
    "Jorhat": "Jorhat",
    "Karbi Anglong": "Karbi Anglong",
    "Majuli": "Majuli",
    "Nagaon": "Nagaon",
    "Sivasagar": "Sivasagar",
    "Sibsagar": "Sivasagar",
    "Sonitpur": "Sonitpur",
    "South Salmara Mancachar": "South Salmara Mancachar",
    "South Salmara-Mankachar": "South Salmara Mancachar",
    "West Karbi Anglong": "West Karbi Anglong",

    # New districts (carry parent district values)
    "Bajali": "Barpeta",
    "Tamulpur": "Baksa"
}

df["district_std"] = df[district_col].replace(district_map)

# =========================
# TARGET DISTRICTS
# =========================
target_districts = [
    "Kokrajhar",
    "Goalpara",
    "Barpeta",
    "Morigaon",
    "Lakhimpur",
    "Dhemaji",
    "Tinsukia",
    "Dibrugarh",
    "Golaghat",
    "Dima Hasao",
    "Cachar",
    "Karimganj",
    "Hailakandi",
    "Bongaigaon",
    "Chirang",
    "Kamrup",
    "Kamrup Metropolitan",
    "Nalbari",
    "Baksa",
    "Darrang",
    "Udalguri",
    "Biswanath",
    "Charaideo",
    "Dhubri",
    "Hojai",
    "Jorhat",
    "Karbi Anglong",
    "Majuli",
    "Nagaon",
    "Sivasagar",
    "Sonitpur",
    "South Salmara Mancachar",
    "West Karbi Anglong"
]

# =========================
# FILTER ASSAM
# =========================
df_assam = df[df["district_std"].isin(target_districts)].copy()

# =========================
# INDICATOR COLUMNS
# =========================
cols = {
    "women_sugar": "Women age 15 years and above wih very high (>160 mg/dl) Blood sugar level23 (%)",
    "men_sugar": "Men (age 15 years and above wih  very high (>160 mg/dl) Blood sugar level23 (%)",
    "women_bp": "Women age 15 years and above wih Moderately or severely elevated blood pressure (Systolic ≥160 mm of Hg and/or Diastolic ≥100 mm of Hg) (%)",
    "men_bp": "Men age 15 years and above wih Moderately or severely elevated blood pressure (Systolic ≥160 mm of Hg and/or Diastolic ≥100 mm of Hg) (%)"
}

# =========================
# VALIDATE COLUMNS
# =========================
missing_cols = [c for c in cols.values() if c not in df_assam.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

# =========================
# CLEAN NUMERIC VALUES
# =========================
for c in cols.values():
    df_assam[c] = pd.to_numeric(df_assam[c], errors="coerce")

# =========================
# COMPUTE NCD INDEX
# =========================
df_assam["pct_ncd"] = df_assam[list(cols.values())].mean(axis=1)

# =========================
# FINAL OUTPUT
# =========================
out = df_assam[
    ["district_std"] + list(cols.values()) + ["pct_ncd"]
].copy()

out = out.rename(columns={
    "district_std": "district",
    cols["women_sugar"]: "women_sugar",
    cols["men_sugar"]: "men_sugar",
    cols["women_bp"]: "women_bp",
    cols["men_bp"]: "men_bp"
})

# =========================
# ORDER DISTRICTS
# =========================
out["district"] = pd.Categorical(
    out["district"],
    categories=target_districts,
    ordered=True
)

out = out.sort_values("district")

# =========================
# VALIDATION
# =========================
matched = out["district"].nunique()
missing = sorted(set(target_districts) - set(out["district"].dropna()))

print("\nMatched districts:", matched)
print("Missing districts:", missing)

assert matched == 33, f"ERROR: Expected 33 districts, found {matched}"

# =========================
# SAVE OUTPUT
# =========================
output_path.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(output_path, index=False)

print("\nSaved to:", output_path)
print(out)

Input: /home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-assam/data_extractor/nfhs/data/factsheet/NFHS_5_India_Districts_Factsheet_Data.xls
Exists: True
Using district column: District

Matched districts: 33
Missing districts: []

Saved to: /home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-assam/data_extractor/nfhs/data/variables/nfhs_ncd_pct.csv
                   district  women_sugar  men_sugar  women_bp  men_bp  pct_ncd
36                Kokrajhar         2.86       4.06      5.34    4.78   4.2600
37                 Goalpara         2.98       3.82      3.28    3.47   3.3875
38                  Barpeta         4.26       4.82      3.94    3.03   4.0125
39                 Morigaon         4.42       4.67      4.87    6.69   5.1625
40                Lakhimpur         3.47       3.43      4.44    5.33   4.1675
41                  Dhemaji         2.22       3.67      4.55    5.35   3.9475
42                 Tinsukia         4.73       8.08      4.66    5.40   5.7175
43               

/tmp/ipykernel_95645/2604228017.py:94: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["district_std"] = df[district_col].replace(district_map)
